# Element-balanced UniversalXAS and Tuned-UniversalXAS FEFF heads

This notebook trains or reuses two balanced UniversalXAS candidates and 56 Tuned-UniversalXAS FEFF candidates from exported 64D features.

- UniversalXAS uses exact element-balanced batches: 4 rows per element and 32 rows total.
- The sampler downsamples without replacement and redraws larger-element subsets each epoch.
- Candidates are `feff_plateau_lr5e-4_do010_seed44` and its seed-45 replicate `feff_plateau_lr5e-4_do010_seed45`.
- The Universal source is selected per element by validation eta only.
- Tuned-UniversalXAS starts from each element's selected Universal source.
- Test evaluation runs for every Universal candidate and only the eight selected tuned checkpoints.
- The notebook computes full-data batches, balanced batches, the update-budget scale, and effective patience after data loading.


## 1. Setup and configuration

The notebook trains or reuses two Universal candidates. Seed 44 is the existing proven setting. Seed 45 is a replicate of that setting. Both candidates use base plateau patience 8, base early-stop patience 60, and scaled effective patience after data loading.

The tuned stage runs the existing seven-setting sweep for every element. Each sweep starts from the Universal candidate selected for that element by validation eta. Test metrics do not select Universal candidates or tuned settings.

The output path is fixed at `SOURCE_RUN / "train_balanced_e2e_universal_tuned_feff"`. Universal artifacts use `universalXAS/{universal_variant}/`. Tuned artifacts use `tunedUniversalXAS/{dataset}/{selected_universal_variant}/{setting}/`.

In [1]:
from __future__ import annotations

from collections.abc import Iterator
from hashlib import sha256
import json
import math
import os
from pathlib import Path

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.data import DataLoader, Sampler, TensorDataset

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule
from omnixas.model.xasblock import XASBlock
from omnixas.model.xasblock_regressor import XASBlockRegressor


def locate_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OmniXAS repo")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

In [2]:
REPO_ROOT = locate_repo_root(Path.cwd().resolve())
FULLTRAINING_ROOT = Path(os.environ.get("OMNIXAS_FULL_TRAINING_ROOT", REPO_ROOT.parent / "fullTrainingCopy072726")).expanduser().resolve()
SOURCE_RUN = FULLTRAINING_ROOT / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42"
FEATURES_DIR = SOURCE_RUN / "features"
SOURCE_ENCODER_CKPT = SOURCE_RUN / "best_e2e_encoder_universal.ckpt"
SOURCE_SETTINGS = SOURCE_RUN / "run_settings.json"
SOURCE_NOTEBOOK = REPO_ROOT / "tutorial_omnixas" / "train_new_universal_and_vasp_encoder.ipynb"
for required in (SOURCE_RUN, FEATURES_DIR, SOURCE_ENCODER_CKPT, SOURCE_SETTINGS, SOURCE_NOTEBOOK):
    if not required.exists(): raise FileNotFoundError(f"Missing required source path: {required}")
RUN_DIR = SOURCE_RUN / "train_balanced_e2e_universal_tuned_feff"
ANALYSIS_DIR = RUN_DIR / "analysis"
ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
FEFF_DATASETS = [f"{element}_FEFF" for element in ELEMENTS]
INPUT_DIM, OUTPUT_DIM = 64, 141
UNIVERSAL_HIDDEN_DIMS = [500, 500, 550]
UNIVERSAL_BATCH_SIZE = 32
UNIVERSAL_ROWS_PER_ELEMENT = 4
UNIVERSAL_SCHEDULER = "ReduceLROnPlateau"
UNIVERSAL_SCHEDULER_MODE = "min"
UNIVERSAL_SCHEDULER_FACTOR = 0.5
UNIVERSAL_SCHEDULER_FREQUENCY = 2
UNIVERSAL_SCHEDULER_MONITOR = "val_median_mse"
UNIVERSAL_MIN_LR = 1e-6
UNIVERSAL_MAX_EPOCHS = 800
UNIVERSAL_CHECK_VAL_EVERY_N_EPOCH = 2
UNIVERSAL_BASE_SCHEDULER_PATIENCE = 8
UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE = 60
UNIVERSAL_DROPOUT = 0.10
UNIVERSAL_LR = 5e-4
_UNIVERSAL_COMMON = {"dropout":UNIVERSAL_DROPOUT,"lr":UNIVERSAL_LR,"scheduler":UNIVERSAL_SCHEDULER,"scheduler_mode":UNIVERSAL_SCHEDULER_MODE,"scheduler_factor":UNIVERSAL_SCHEDULER_FACTOR,"base_scheduler_patience":UNIVERSAL_BASE_SCHEDULER_PATIENCE,"min_lr":UNIVERSAL_MIN_LR,"scheduler_frequency":UNIVERSAL_SCHEDULER_FREQUENCY,"max_epochs":UNIVERSAL_MAX_EPOCHS,"base_early_stopping_patience":UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE,"check_val_every_n_epoch":UNIVERSAL_CHECK_VAL_EVERY_N_EPOCH}
UNIVERSAL_CANDIDATES = [{"name":name,"seed":seed,**_UNIVERSAL_COMMON} for name,seed in (("feff_plateau_lr5e-4_do010_seed44",44),("feff_plateau_lr5e-4_do010_seed45",45))]
UNIVERSAL_CANDIDATE_NAMES = [c["name"] for c in UNIVERSAL_CANDIDATES]
UNIVERSAL_SAMPLER_SEED = int(UNIVERSAL_CANDIDATES[0]["seed"])
if len(set(UNIVERSAL_CANDIDATE_NAMES)) != len(UNIVERSAL_CANDIDATE_NAMES): raise ValueError("Universal candidate names must be unique")
TUNED_HIDDEN_DIMS = [500, 500, 550]
TUNED_SCHEDULER = "cosine"
TUNED_ETA_MIN = 1e-6
TUNED_TASK_BATCH = {"Ti_FEFF":32,"V_FEFF":32,"Cr_FEFF":32,"Mn_FEFF":64,"Fe_FEFF":64,"Co_FEFF":32,"Ni_FEFF":32,"Cu_FEFF":32}
TUNED_SWEEP = [
 {"name":"cosT500_lr3e-4_do0p1_p60","seed":145,"dropout":0.10,"lr":3e-4,"cosine_t":500,"epochs":1000,"patience":60},
 #{"name":"cosT500_lr2e-4_do0p1_p60","seed":145,"dropout":0.10,"lr":2e-4,"cosine_t":500,"epochs":1000,"patience":60},
 #{"name":"cosT500_lr4e-4_do0p1_p60","seed":145,"dropout":0.10,"lr":4e-4,"cosine_t":500,"epochs":1000,"patience":60},
 {"name":"cosT500_lr3e-4_do0p05_p60","seed":145,"dropout":0.05,"lr":3e-4,"cosine_t":500,"epochs":1000,"patience":60},
 {"name":"cosT500_lr3e-4_do0p15_p60","seed":145,"dropout":0.15,"lr":3e-4,"cosine_t":500,"epochs":1000,"patience":60},
 {"name":"cosT750_lr3e-4_do0p1_p60","seed":145,"dropout":0.10,"lr":3e-4,"cosine_t":750,"epochs":1000,"patience":60},
 {"name":"cosT750_lr4e-4_do0p1_p60","seed":145,"dropout":0.10,"lr":4e-4,"cosine_t":750,"epochs":1000,"patience":60},]
if len({c["name"] for c in TUNED_SWEEP}) != len(TUNED_SWEEP): raise ValueError("Tuned sweep setting names must be unique")
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if UNIVERSAL_BATCH_SIZE != len(ELEMENTS) * UNIVERSAL_ROWS_PER_ELEMENT: raise ValueError("Universal batch size mismatch")
pl.seed_everything(UNIVERSAL_CANDIDATES[0]["seed"], workers=True)
print(f"Source run: {SOURCE_RUN}")
print(f"Output run: {RUN_DIR}")
print(f"Device: {DEVICE}")

Seed set to 44


Source run: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
Output run: /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff
Device: cuda:0


## 2. Data preflight

Each FEFF feature, target, and material/site ID file must have the same row count. The published train, validation, and test material split isolation is checked within each element. Cross-element overlap is allowed because the published per-element split files overlap across elements.

In [3]:
MATERIAL_IDS_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"


def load_split(dataset: str) -> MLSplits:
    parts: dict[str, MLData] = {}
    parsed_material_ids: dict[str, list[str]] = {}
    for split_name in ("train", "val", "test"):
        x_path = FEATURES_DIR / f"{dataset}_{split_name}_X.txt"
        y_path = FEATURES_DIR / f"{dataset}_{split_name}_y.txt"
        ids_path = MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt"
        if not x_path.is_file() or not y_path.is_file() or not ids_path.is_file():
            raise FileNotFoundError(
                f"Missing {dataset} {split_name}: {x_path}, {y_path}, {ids_path}"
            )

        X = np.atleast_2d(np.loadtxt(x_path, dtype=np.float32))
        y = np.atleast_2d(np.loadtxt(y_path, dtype=np.float32))
        if X.ndim != 2 or y.ndim != 2:
            raise ValueError(
                f"Invalid dimensions for {dataset} {split_name}: "
                f"X={X.shape}, y={y.shape}"
            )
        if X.shape[0] != y.shape[0] or X.shape[1] != INPUT_DIM or y.shape[1] != OUTPUT_DIM:
            raise ValueError(f"Invalid {dataset} {split_name}: X={X.shape}, y={y.shape}")
        if not np.isfinite(X).all() or not np.isfinite(y).all():
            raise ValueError(f"Non-finite values in {dataset} {split_name}")

        id_rows = [
            line.strip()
            for line in ids_path.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        if len(id_rows) != X.shape[0] or len(id_rows) != y.shape[0]:
            raise ValueError(
                f"Material ID row count mismatch for {dataset} {split_name}: "
                f"IDs={len(id_rows)}, X={X.shape[0]}, y={y.shape[0]}"
            )

        parsed_ids = []
        for row in id_rows:
            material_id_parts = row.rsplit("_", 1)
            if len(material_id_parts) != 2 or not material_id_parts[0] or not material_id_parts[1]:
                raise ValueError(f"Invalid material/site row in {ids_path}: {row!r}")
            parsed_ids.append(material_id_parts[0])
        parsed_material_ids[split_name] = parsed_ids
        parts[split_name] = MLData(X=X, y=y)

    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = set(parsed_material_ids[left]) & set(parsed_material_ids[right])
        if overlap:
            raise ValueError(
                f"Material IDs overlap for {dataset}: {left}/{right}: {sorted(overlap)[:5]}"
            )
    return MLSplits(**parts)


splits = {dataset: load_split(dataset) for dataset in FEFF_DATASETS}

In [4]:
def combine_element_split(
    split_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_blocks: list[np.ndarray] = []
    y_blocks: list[np.ndarray] = []
    element_blocks: list[np.ndarray] = []
    for element_index, dataset in enumerate(FEFF_DATASETS):
        part = getattr(splits[dataset], split_name)
        X_blocks.append(part.X)
        y_blocks.append(part.y)
        element_blocks.append(np.full(len(part), element_index, dtype=np.int64))
    return np.concatenate(X_blocks), np.concatenate(y_blocks), np.concatenate(element_blocks)


universal_train_X, universal_train_y, train_element_codes = combine_element_split("train")
universal_val_X, universal_val_y, val_element_codes = combine_element_split("val")
universal_test_X, universal_test_y, test_element_codes = combine_element_split("test")
train_indices_by_element = {
    element_index: np.flatnonzero(train_element_codes == element_index).tolist()
    for element_index in range(len(ELEMENTS))
}
val_baselines = np.array(
    [
        np.median(np.mean((split.val.y - split.train.y.mean(axis=0)) ** 2, axis=1))
        for split in (splits[dataset] for dataset in FEFF_DATASETS)
    ],
    dtype=np.float32,
)
if not np.isfinite(val_baselines).all() or (val_baselines <= 0).any():
    raise ValueError(f"Invalid validation baseline medians: {val_baselines}")

shape_table = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "train": len(splits[dataset].train),
            "val": len(splits[dataset].val),
            "test": len(splits[dataset].test),
            "X_dim": splits[dataset].train.X.shape[1],
            "y_dim": splits[dataset].train.y.shape[1],
        }
        for dataset in FEFF_DATASETS
    ]
)
display(shape_table)
print(
    f"Universal rows: train={len(universal_train_y)}, "
    f"val={len(universal_val_y)}, test={len(universal_test_y)}"
)

,dataset,train,val,test,X_dim,y_dim
0,Ti_FEFF,5140,641,641,64,141
1,V_FEFF,8653,1080,1080,64,141
2,Cr_FEFF,2457,305,305,64,141
3,Mn_FEFF,13644,1704,1704,64,141
4,Fe_FEFF,9657,1205,1205,64,141
5,Co_FEFF,8605,1074,1074,64,141
6,Ni_FEFF,3471,432,432,64,141
7,Cu_FEFF,3340,416,416,64,141


Universal rows: train=54967, val=6857, test=6857


## 3. Sampler and update-budget configuration

The sampler shuffles each element pool, selects `UNIVERSAL_ROWS_PER_ELEMENT * n_batches` rows without replacement, and forms exact 32-row batches. The sampler increments its epoch inside `__iter__`, so larger-element subsets change between training epochs. The notebook creates the real sampler without iterating it before training. It computes full-data batches, balanced batches, the update-budget scale, and effective scheduler and early-stopping patience from the loaded data.


In [5]:
class ElementStratifiedBatchSampler(Sampler[list[int]]):
    def __init__(
        self,
        indices_by_element: dict[int, list[int]],
        per_element: int,
        seed: int,
    ) -> None:
        if not indices_by_element:
            raise ValueError("Element index pools cannot be empty")
        if per_element < 1:
            raise ValueError("per_element must be positive")

        self.indices_by_element = {
            int(key): list(value) for key, value in indices_by_element.items()
        }
        self.element_order = sorted(self.indices_by_element)
        self.per_element = int(per_element)
        self.seed = int(seed)
        self.epoch = 0
        counts = [len(self.indices_by_element[key]) for key in self.element_order]
        self.n_batches = min(counts) // self.per_element
        if self.n_batches < 1:
            raise ValueError(f"Each element needs at least {self.per_element} rows")

    def __len__(self) -> int:
        return self.n_batches

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        selected = {
            element: rng.permutation(self.indices_by_element[element])[
                : self.n_batches * self.per_element
            ]
            for element in self.element_order
        }
        batches: list[list[int]] = []
        for batch_number in range(self.n_batches):
            batch = np.concatenate(
                [
                    selected[element][
                        batch_number * self.per_element : (batch_number + 1) * self.per_element
                    ]
                    for element in self.element_order
                ]
            )
            rng.shuffle(batch)
            batches.append(batch.tolist())
        return iter(batches)


In [6]:
real_sampler = ElementStratifiedBatchSampler(
    train_indices_by_element,
    UNIVERSAL_ROWS_PER_ELEMENT,
    UNIVERSAL_SAMPLER_SEED,
)
UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH = math.ceil(len(universal_train_y) / UNIVERSAL_BATCH_SIZE)
UNIVERSAL_BALANCED_BATCHES_PER_EPOCH = len(real_sampler)
if UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH < 1:
    raise ValueError("Universal training data must provide at least one full-data batch")
if UNIVERSAL_BALANCED_BATCHES_PER_EPOCH < 1:
    raise ValueError("Balanced sampler must provide at least one batch")
UNIVERSAL_UPDATE_BUDGET_SCALE = UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH / UNIVERSAL_BALANCED_BATCHES_PER_EPOCH
UNIVERSAL_EFFECTIVE_SCHEDULER_PATIENCE = round(UNIVERSAL_BASE_SCHEDULER_PATIENCE * UNIVERSAL_UPDATE_BUDGET_SCALE)
UNIVERSAL_EFFECTIVE_EARLY_STOPPING_PATIENCE = round(UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE * UNIVERSAL_UPDATE_BUDGET_SCALE)


## 4. Source manifest

The manifest records source hashes, both Universal candidate configurations, the per-dataset validation-only Universal selection policy, the tuned sweep, and all analysis paths.

In [7]:
manifest_path = RUN_DIR / "settings_source_manifest.json"
sampler_info_path = RUN_DIR / "sampler_info.json"
if (RUN_DIR / "RUN_COMPLETE.json").exists(): raise RuntimeError(f"Run is already complete: {RUN_DIR / 'RUN_COMPLETE.json'}")
existing_manifest: dict[str, object] = {}
current_source_encoder_sha256 = sha256_file(SOURCE_ENCODER_CKPT)
if RUN_DIR.exists():
    if not RUN_DIR.is_dir(): raise FileExistsError(f"Output root is not a directory: {RUN_DIR}")
    if any(RUN_DIR.iterdir()):
        if not manifest_path.is_file(): raise FileNotFoundError(f"Non-empty output root requires settings_source_manifest.json: {manifest_path}")
        existing_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing_manifest.get("source_encoder_checkpoint_sha256") != current_source_encoder_sha256:
            raise ValueError("Source encoder checkpoint hash mismatch")
        saved_candidates = existing_manifest.get("universal_candidates")
        if saved_candidates is not None:
            saved_names = [x.get("name") if isinstance(x, dict) else x for x in saved_candidates]
            if saved_names != UNIVERSAL_CANDIDATE_NAMES:
                print("Saved universal_candidates differ from current notebook settings. Existing checkpoint directories will be validated later by find_best_checkpoint; the current manifest will be rewritten.")
        saved_variant = existing_manifest.get("universal_variant")
        if saved_variant is not None and saved_variant not in UNIVERSAL_CANDIDATE_NAMES:
            print(f"Saved universal_variant {saved_variant!r} differs from current notebook settings. Existing checkpoint directories will be validated later by find_best_checkpoint; the current manifest will be rewritten.")
        saved_tuned_sweep = existing_manifest.get("tuned_sweep")
        if saved_tuned_sweep != TUNED_SWEEP:
            print(f"Saved manifest tuned_sweep differs from current notebook settings ({saved_tuned_sweep!r} != {TUNED_SWEEP!r}). It will be replaced by the current notebook settings.")
        print(f"Validated incomplete run. Resuming: {RUN_DIR}")
else: RUN_DIR.mkdir(parents=True, exist_ok=False)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
source_files = [SOURCE_ENCODER_CKPT, SOURCE_SETTINGS, SOURCE_NOTEBOOK]
for dataset in FEFF_DATASETS:
    for split_name in ("train", "val", "test"):
        source_files.extend([FEATURES_DIR / f"{dataset}_{split_name}_X.txt", FEATURES_DIR / f"{dataset}_{split_name}_y.txt", MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt"])
source_files = sorted({x for x in source_files if x.is_file()})
universal_manifest_candidates = [{**c, "effective_scheduler_patience": UNIVERSAL_EFFECTIVE_SCHEDULER_PATIENCE, "effective_early_stopping_patience": UNIVERSAL_EFFECTIVE_EARLY_STOPPING_PATIENCE} for c in UNIVERSAL_CANDIDATES]
universal_selection_policy = "Select the highest validation eta per FEFF dataset across the two Universal candidates. Do not use test eta for selection."
analysis_files = {"universal":str(ANALYSIS_DIR/"universal_eval.csv"),"universal_selected_by_dataset":str(ANALYSIS_DIR/"universal_selected_by_dataset.csv"),"tuned_validation_candidates":str(ANALYSIS_DIR/"tuned_validation_candidates.csv"),"tuned":str(ANALYSIS_DIR/"tuned_eval.csv"),"combined":str(ANALYSIS_DIR/"evaluation_summary.csv")}
source_manifest = {
 "run_label":"balanced_new_scheduler_universal_tuned_feff","source_run":str(SOURCE_RUN),"source_encoder_checkpoint":str(SOURCE_ENCODER_CKPT),"source_encoder_checkpoint_sha256":current_source_encoder_sha256,
 "source_settings":str(SOURCE_SETTINGS),"source_settings_sha256":sha256_file(SOURCE_SETTINGS),"source_settings_used_for_training":False,"source_notebook":str(SOURCE_NOTEBOOK),"source_notebook_sha256":sha256_file(SOURCE_NOTEBOOK),"source_not_retrained":True,
 "comparison_type":"controlled balanced-sampler run with two Universal candidates","balanced_sampler_is_intended_training_difference":True,"datasets":FEFF_DATASETS,"input_dim":INPUT_DIM,"output_dim":OUTPUT_DIM,"feature_scale":1000,"split_membership_note":"Published per-element split membership is preserved.",
 "source_files":[{"path":str(x),"sha256":sha256_file(x)} for x in source_files],"output_run":str(RUN_DIR),"analysis_dir":str(ANALYSIS_DIR),"sampler_info":str(sampler_info_path),"universal_candidates":universal_manifest_candidates,"universal_selection_policy":universal_selection_policy,
 "universal_hidden_dims":UNIVERSAL_HIDDEN_DIMS,"universal_batch_size":UNIVERSAL_BATCH_SIZE,"universal_rows_per_element":UNIVERSAL_ROWS_PER_ELEMENT,"universal_scheduler":UNIVERSAL_SCHEDULER,"universal_scheduler_mode":UNIVERSAL_SCHEDULER_MODE,"universal_scheduler_factor":UNIVERSAL_SCHEDULER_FACTOR,"universal_min_lr":UNIVERSAL_MIN_LR,"universal_scheduler_frequency":UNIVERSAL_SCHEDULER_FREQUENCY,"universal_scheduler_monitor":UNIVERSAL_SCHEDULER_MONITOR,"universal_max_epochs":UNIVERSAL_MAX_EPOCHS,"universal_base_scheduler_patience":UNIVERSAL_BASE_SCHEDULER_PATIENCE,"universal_effective_scheduler_patience":UNIVERSAL_EFFECTIVE_SCHEDULER_PATIENCE,"universal_base_early_stopping_patience":UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE,"universal_effective_early_stopping_patience":UNIVERSAL_EFFECTIVE_EARLY_STOPPING_PATIENCE,
 "tuned_sweep":TUNED_SWEEP,"tuned_hidden_dims":TUNED_HIDDEN_DIMS,"tuned_scheduler":TUNED_SCHEDULER,"tuned_eta_min":TUNED_ETA_MIN,"tuned_task_batch":TUNED_TASK_BATCH,"tuned_training_monitor":"val_median_mse","tuned_selection_metric":"val_eta","tuned_validation_only_selection":True,
 "universal_full_data_batches_per_epoch":UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH,"universal_balanced_batches_per_epoch":UNIVERSAL_BALANCED_BATCHES_PER_EPOCH,"universal_update_budget_scale":UNIVERSAL_UPDATE_BUDGET_SCALE,"universal_patience_scaling_reason":"Scale patience by full-data batches divided by strict balanced batches to preserve optimizer-step-equivalent patience.","universal_epoch_batches":UNIVERSAL_BALANCED_BATCHES_PER_EPOCH,
 "train_counts":{e:len(splits[f"{e}_FEFF"].train) for e in ELEMENTS},"validation_and_test_rows_retained":True,"inverse_count_loss_weights":False,"weighted_random_sampler":False,"evaluation_files":analysis_files}
if "migrated_from" in existing_manifest: source_manifest["migrated_from"] = existing_manifest["migrated_from"]
manifest_path.write_text(json.dumps(source_manifest,indent=2),encoding="utf-8")
sampler_info_path.write_text(
    json.dumps(
        {
            "sampler": "ElementStratifiedBatchSampler",
            "seed": UNIVERSAL_SAMPLER_SEED,
            "batch_size": UNIVERSAL_BATCH_SIZE,
            "per_element": UNIVERSAL_ROWS_PER_ELEMENT,
            "full_data_batches_per_epoch": UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH,
            "balanced_batches_per_epoch": UNIVERSAL_BALANCED_BATCHES_PER_EPOCH,
            "update_budget_scale": UNIVERSAL_UPDATE_BUDGET_SCALE,
        },
        indent=2,
    ),
    encoding="utf-8",
)

Saved manifest tuned_sweep differs from current notebook settings ([{'name': 'cosT500_lr3e-4_do0p1_p60', 'seed': 145, 'dropout': 0.1, 'lr': 0.0003, 'cosine_t': 500, 'epochs': 1000, 'patience': 60}, {'name': 'cosT500_lr2e-4_do0p1_p60', 'seed': 145, 'dropout': 0.1, 'lr': 0.0002, 'cosine_t': 500, 'epochs': 1000, 'patience': 60}, {'name': 'cosT500_lr4e-4_do0p1_p60', 'seed': 145, 'dropout': 0.1, 'lr': 0.0004, 'cosine_t': 500, 'epochs': 1000, 'patience': 60}, {'name': 'cosT500_lr3e-4_do0p05_p60', 'seed': 145, 'dropout': 0.05, 'lr': 0.0003, 'cosine_t': 500, 'epochs': 1000, 'patience': 60}, {'name': 'cosT500_lr3e-4_do0p15_p60', 'seed': 145, 'dropout': 0.15, 'lr': 0.0003, 'cosine_t': 500, 'epochs': 1000, 'patience': 60}, {'name': 'cosT750_lr3e-4_do0p1_p60', 'seed': 145, 'dropout': 0.1, 'lr': 0.0003, 'cosine_t': 750, 'epochs': 1000, 'patience': 60}, {'name': 'cosT750_lr4e-4_do0p1_p60', 'seed': 145, 'dropout': 0.1, 'lr': 0.0004, 'cosine_t': 750, 'epochs': 1000, 'patience': 60}] != [{'name': 'cosT

222

## 5. Universal data and module

Both Universal candidates use the same balanced data module and head. Only the seed differs. The notebook scales base patience by the full-data to balanced-batch update budget.


In [8]:
class UniversalBalancedData(pl.LightningDataModule):
    def __init__(
        self,
        train_X: np.ndarray,
        train_y: np.ndarray,
        train_element_codes: np.ndarray,
        val_X: np.ndarray,
        val_y: np.ndarray,
        val_element_codes: np.ndarray,
        sampler: ElementStratifiedBatchSampler,
    ) -> None:
        super().__init__()
        self.train = TensorDataset(
            torch.as_tensor(train_X, dtype=torch.float32),
            torch.as_tensor(train_y, dtype=torch.float32),
            torch.as_tensor(train_element_codes, dtype=torch.long),
        )
        self.val = TensorDataset(
            torch.as_tensor(val_X, dtype=torch.float32),
            torch.as_tensor(val_y, dtype=torch.float32),
            torch.as_tensor(val_element_codes, dtype=torch.long),
        )
        self.sampler = sampler

    def train_dataloader(self) -> DataLoader:
        return DataLoader(self.train, batch_sampler=self.sampler)

    def val_dataloader(self) -> DataLoader:
        return DataLoader(self.val, batch_size=1024, shuffle=False)

In [9]:
class BalancedUniversalModule(PlModule):
    def __init__(
        self,
        model: XASBlock,
        val_baselines: np.ndarray,
        **kwargs,
    ) -> None:
        super().__init__(model=model, **kwargs)
        self.register_buffer(
            "val_baselines",
            torch.as_tensor(val_baselines, dtype=torch.float32),
        )
        self.val_mses: list[torch.Tensor] = []
        self.val_elements: list[torch.Tensor] = []

    def training_step(self, batch: tuple[torch.Tensor, ...], batch_idx: int) -> torch.Tensor:
        x, y, _ = batch
        return self.logged_loss("train_loss", y, self.model(x))

    def on_validation_epoch_start(self) -> None:
        self.val_mses = []
        self.val_elements = []

    def validation_step(
        self,
        batch: tuple[torch.Tensor, ...],
        batch_idx: int,
    ) -> torch.Tensor:
        x, y, elements = batch
        y_pred = self.model(x)
        sample_mse = torch.mean((y - y_pred) ** 2, dim=1)
        self.val_mses.append(sample_mse.detach())
        self.val_elements.append(elements.detach())
        return self.logged_loss("val_loss", y, y_pred)

    def on_validation_epoch_end(self) -> None:
        if self.trainer.sanity_checking:
            return
        if not self.val_mses:
            raise RuntimeError("Validation produced no rows")
        mses = torch.cat(self.val_mses)
        elements = torch.cat(self.val_elements)
        self.log(
            "val_median_mse",
            torch.quantile(mses, 0.5),
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )
        relative_medians = []
        for element_index, element in enumerate(ELEMENTS):
            mask = elements == element_index
            if not mask.any():
                raise RuntimeError(f"Validation has no rows for {element}")
            median_mse = torch.quantile(mses[mask], 0.5)
            relative_medians.append(median_mse / self.val_baselines[element_index])
        self.log(
            "val_balanced_rel_mse",
            torch.stack(relative_medians).mean(),
            on_step=False,
            on_epoch=True,
            prog_bar=True,
        )

## 6. Universal training

The notebook reuses a candidate when its directory contains exactly one best checkpoint. It trains a missing candidate. A non-empty candidate directory without exactly one best checkpoint fails. The checkpoint monitor is aggregate validation `val_median_mse`. Universal source selection occurs later by validation eta per dataset.

In [10]:
def find_best_checkpoint(setting_dir: Path) -> Path | None:
    run_root = RUN_DIR.resolve(); setting_root = setting_dir.resolve()
    if not setting_root.is_relative_to(run_root): raise ValueError(f"Checkpoint setting is outside RUN_DIR: {setting_dir}")
    if not setting_dir.exists(): return None
    if not setting_dir.is_dir(): raise FileExistsError(f"Checkpoint setting is not a directory: {setting_dir}")
    if not any(setting_dir.iterdir()): return None
    candidates = sorted(x for x in setting_dir.glob("best*.ckpt") if x.is_file())
    if len(candidates) != 1: raise RuntimeError(f"Expected exactly one best*.ckpt in non-empty setting directory {setting_dir}, found {len(candidates)}")
    checkpoint = candidates[0].resolve()
    if not checkpoint.is_relative_to(run_root): raise ValueError(f"Checkpoint is outside RUN_DIR: {checkpoint}")
    return checkpoint

print(f"Universal budget: full={UNIVERSAL_FULL_DATA_BATCHES_PER_EPOCH}, balanced={UNIVERSAL_BALANCED_BATCHES_PER_EPOCH}, scale={UNIVERSAL_UPDATE_BUDGET_SCALE:.3f}; patience={UNIVERSAL_EFFECTIVE_SCHEDULER_PATIENCE}/{UNIVERSAL_EFFECTIVE_EARLY_STOPPING_PATIENCE}")
universal_checkpoints: dict[str, Path] = {}
for candidate in UNIVERSAL_CANDIDATES:
    variant = candidate["name"]; out = RUN_DIR / "universalXAS" / variant
    checkpoint = find_best_checkpoint(out)
    if checkpoint is not None:
        print(f"Reusing UniversalXAS checkpoint ({variant}): {checkpoint}"); universal_checkpoints[variant] = checkpoint; continue
    pl.seed_everything(int(candidate["seed"]), workers=True)
    XASBlock.DROPOUT = float(candidate["dropout"])
    module = BalancedUniversalModule(XASBlock(INPUT_DIM, UNIVERSAL_HIDDEN_DIMS, OUTPUT_DIM), val_baselines, lr=float(candidate["lr"]), lr_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau, lr_scheduler_kwargs={"mode":candidate["scheduler_mode"],"factor":float(candidate["scheduler_factor"]),"patience":UNIVERSAL_EFFECTIVE_SCHEDULER_PATIENCE,"min_lr":float(candidate["min_lr"])}, lr_scheduler_interval="epoch", lr_scheduler_frequency=int(candidate["scheduler_frequency"]), lr_scheduler_monitor=UNIVERSAL_SCHEDULER_MONITOR)
    out.mkdir(parents=True, exist_ok=True)
    callback = ModelCheckpoint(dirpath=str(out), filename=f"best-{variant}-{{epoch:03d}}-{{val_median_mse:.5f}}", monitor=UNIVERSAL_SCHEDULER_MONITOR, mode="min", save_top_k=1, save_last=True)
    trainer = pl.Trainer(max_epochs=int(candidate["max_epochs"]), accelerator="auto", devices=1, check_val_every_n_epoch=int(candidate["check_val_every_n_epoch"]), callbacks=[EarlyStopping(monitor=UNIVERSAL_SCHEDULER_MONITOR, patience=UNIVERSAL_EFFECTIVE_EARLY_STOPPING_PATIENCE, mode="min"), callback], logger=CSVLogger(save_dir=str(out), name="logs"), log_every_n_steps=1)
    sampler = ElementStratifiedBatchSampler(train_indices_by_element, UNIVERSAL_ROWS_PER_ELEMENT, int(candidate["seed"]))
    data = UniversalBalancedData(universal_train_X, universal_train_y, train_element_codes, universal_val_X, universal_val_y, val_element_codes, sampler)
    trainer.fit(module, datamodule=data)
    checkpoint = find_best_checkpoint(out)
    if checkpoint is None: raise RuntimeError(f"UniversalXAS training finished without a best checkpoint: {variant}")
    universal_checkpoints[variant] = checkpoint
    print(f"UniversalXAS checkpoint ({variant}): {checkpoint}")
if set(universal_checkpoints) != set(UNIVERSAL_CANDIDATE_NAMES): raise RuntimeError("Missing Universal candidate checkpoint")

Seed set to 45


Universal budget: full=1718, balanced=614, scale=2.798; patience=22/168
Reusing UniversalXAS checkpoint (feff_plateau_lr5e-4_do010_seed44): /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff/universalXAS/feff_plateau_lr5e-4_do010_seed44/best-feff_plateau_lr5e-4_do010_seed44-epoch=743-val_median_mse=0.00221.ckpt


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA GeForce RTX 4070 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff/universalXAS/feff_plateau_lr5e-4_do010_see

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

`Trainer.fit` stopped: `max_epochs=800` reached.


UniversalXAS checkpoint (feff_plateau_lr5e-4_do010_seed45): /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff/universalXAS/feff_plateau_lr5e-4_do010_seed45/best-feff_plateau_lr5e-4_do010_seed45-epoch=711-val_median_mse=0.00221.ckpt


## 7. Evaluation helpers and Universal evaluation

The notebook evaluates validation and test for every Universal candidate. Universal source selection uses validation eta only. Tuned candidates use validation only, and selected tuned checkpoints receive test evaluation after selection.

In [11]:
def head_state_from_checkpoint(checkpoint: Path) -> dict[str, torch.Tensor]:
    # These checkpoints are trusted local Lightning artifacts.
    checkpoint_data = torch.load(checkpoint, map_location="cpu")
    state = checkpoint_data.get("state_dict")
    if state is None:
        raise ValueError(f"Missing state_dict in {checkpoint}")
    head_state = {key.removeprefix("model."): value for key, value in state.items() if key.startswith("model.")}
    if not head_state:
        raise ValueError(f"Missing XASBlock state in {checkpoint}")
    return head_state


def load_checkpoint_model(checkpoint: Path, *, hidden_dims: list[int], dropout: float) -> tuple[XASBlock, dict[str, torch.Tensor]]:
    XASBlock.DROPOUT = dropout
    head_state = head_state_from_checkpoint(checkpoint)
    model = XASBlock(INPUT_DIM, hidden_dims, OUTPUT_DIM)
    model.load_state_dict(head_state, strict=True)
    model.to(DEVICE).eval()
    return model, head_state


def predict_model(model: XASBlock, X: np.ndarray, batch_size: int = 1024) -> np.ndarray:
    loader = DataLoader(TensorDataset(torch.as_tensor(X, dtype=torch.float32)), batch_size=batch_size)
    predictions = []
    with torch.inference_mode():
        for (batch,) in loader:
            predictions.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(predictions)

In [12]:
def split_metrics(
    split: MLSplits,
    split_name: str,
    predictions: np.ndarray,
) -> dict[str, float]:
    target = getattr(split, split_name).y
    train_mean = split.train.y.mean(axis=0, keepdims=True)
    median_mse = float(np.median(np.mean((target - predictions) ** 2, axis=1)))
    baseline_median_mse = float(
        np.median(np.mean((target - train_mean) ** 2, axis=1))
    )
    if baseline_median_mse <= 0 or median_mse <= 0:
        raise ValueError(
            f"Invalid MSE for {split_name}: "
            f"model={median_mse}, baseline={baseline_median_mse}"
        )
    return {
        f"{split_name}_median_mse": median_mse,
        f"{split_name}_baseline_median_mse": baseline_median_mse,
        f"{split_name}_eta": baseline_median_mse / median_mse,
    }


def evaluate_loaded_checkpoint(
    model: XASBlock,
    checkpoint: Path,
    model_name: str,
    split_by_dataset: dict[str, MLSplits],
    extra_fields: dict[str, object] | None = None,
    split_names: tuple[str, ...] = ("val", "test"),
) -> pd.DataFrame:
    allowed_split_names = {"val", "test"}
    requested_split_names = tuple(split_names)
    unsupported = set(requested_split_names) - allowed_split_names
    if not requested_split_names or unsupported:
        raise ValueError(f"Unsupported requested split names: {sorted(unsupported)}")
    rows = []
    datasets = [dataset for dataset in FEFF_DATASETS if dataset in split_by_dataset]
    for dataset in datasets:
        split = split_by_dataset[dataset]
        row = {
            "model": model_name,
            "dataset": dataset,
            "element": dataset.removesuffix("_FEFF"),
            "checkpoint": str(checkpoint),
            "checkpoint_sha256": sha256_file(checkpoint),
        }
        if extra_fields:
            row.update(extra_fields)
        predictions = {
            split_name: predict_model(model, getattr(split, split_name).X)
            for split_name in requested_split_names
        }
        for split_name in requested_split_names:
            row.update(split_metrics(split, split_name, predictions[split_name]))
        rows.append(row)
    return pd.DataFrame(rows)

In [13]:
universal_eval_frames = []
universal_head_states: dict[str, dict[str, torch.Tensor]] = {}
for candidate in UNIVERSAL_CANDIDATES:
    variant = candidate["name"]; checkpoint = universal_checkpoints[variant]
    model, state = load_checkpoint_model(checkpoint, hidden_dims=UNIVERSAL_HIDDEN_DIMS, dropout=float(candidate["dropout"]))
    universal_head_states[variant] = state
    frame = evaluate_loaded_checkpoint(model, checkpoint, "UniversalXAS", splits, extra_fields={"universal_variant":variant,"universal_seed":int(candidate["seed"])}, split_names=("val","test"))
    frame["val_balanced_rel_mse_by_element"] = frame["val_median_mse"] / frame["val_baseline_median_mse"]
    frame["val_balanced_rel_mse"] = frame["val_balanced_rel_mse_by_element"]
    frame["val_balanced_rel_mse_macro"] = frame["val_balanced_rel_mse_by_element"].mean()
    universal_eval_frames.append(frame)
    del model
universal_eval = pd.concat(universal_eval_frames, ignore_index=True).sort_values(["dataset","universal_variant"])
universal_eval.to_csv(ANALYSIS_DIR / "universal_eval.csv", index=False)
selected_universal_rows = []; selected_universal_by_dataset: dict[str, dict[str, object]] = {}
for dataset in FEFF_DATASETS:
    candidates = universal_eval[universal_eval["dataset"] == dataset]
    if len(candidates) != len(UNIVERSAL_CANDIDATES): raise RuntimeError(f"Expected every Universal candidate for {dataset}")
    selected = candidates.loc[candidates["val_eta"].idxmax()].to_dict()
    selected_universal_rows.append({"dataset":dataset,"element":selected["element"],"selected_universal_variant":selected["universal_variant"],"selected_universal_seed":int(selected["universal_seed"]),"selected_universal_checkpoint":selected["checkpoint"],"selected_universal_checkpoint_sha256":selected["checkpoint_sha256"],"selected_val_eta":float(selected["val_eta"]),"selected_test_eta":float(selected["test_eta"]),"selected_val_median_mse":float(selected["val_median_mse"]),"selected_test_median_mse":float(selected["test_median_mse"]),"selected_val_balanced_rel_mse":float(selected["val_balanced_rel_mse"])})
    selected_universal_by_dataset[dataset] = selected
universal_selected_by_dataset = pd.DataFrame(selected_universal_rows).sort_values("dataset")
universal_selected_by_dataset.to_csv(ANALYSIS_DIR / "universal_selected_by_dataset.csv", index=False)

## 8. Tuned training and validation selection

Each of the seven sweep settings starts from the Universal checkpoint selected for that dataset by validation eta. The 56 candidates evaluate validation only. The highest `val_eta` selects one checkpoint per dataset. Test evaluation runs once for each selected checkpoint.

Output uses `tunedUniversalXAS/{dataset}/{selected_universal_variant}/{setting}/`. Each candidate writes selected Universal variant, checkpoint, hash, and setting provenance.

In [ ]:
tuned_validation_rows = []
for dataset in FEFF_DATASETS:
    split = splits[dataset]
    selected_source = selected_universal_by_dataset[dataset]
    selected_variant = str(selected_source["universal_variant"])
    selected_universal_checkpoint = Path(str(selected_source["checkpoint"])).resolve()
    selected_universal_checkpoint_sha256 = str(selected_source["checkpoint_sha256"])
    selected_head_state = universal_head_states[selected_variant]
    for cfg in TUNED_SWEEP:
        setting = cfg["name"]
        out = RUN_DIR / "tunedUniversalXAS" / dataset / selected_variant / setting
        tuned_checkpoint = find_best_checkpoint(out)
        if tuned_checkpoint is None:
            out.mkdir(parents=True, exist_ok=True)
            pl.seed_everything(int(cfg["seed"]), workers=True)
            XASBlock.DROPOUT = float(cfg["dropout"])
            tuned = XASBlockRegressor(
                directory=str(out),
                overwrite_save_dir=False,
                input_dim=INPUT_DIM,
                output_dim=OUTPUT_DIM,
                hidden_dims=TUNED_HIDDEN_DIMS,
                batch_size=TUNED_TASK_BATCH[dataset],
                max_epochs=int(cfg["epochs"]),
                early_stopping_patience=int(cfg["patience"]),
                initial_lr=float(cfg["lr"]),
                min_lr=TUNED_ETA_MIN,
                use_lr_finder=False,
                use_early_stopping=True,
                monitor_metric="val_median_mse",
                shuffle=True,
                lr_scheduler=TUNED_SCHEDULER,
                cosine_t_max=int(cfg["cosine_t"]),
                cosine_eta_min=TUNED_ETA_MIN,
            )
            tuned.model.model.load_state_dict(selected_head_state, strict=True)
            tuned.fit(split)
            tuned_checkpoint = Path(tuned.cfg.fetch_checkpoint("best")).resolve()
        else:
            print(f"Reusing tuned checkpoint ({dataset}/{setting}): {tuned_checkpoint}")
        tuned_checkpoint_sha256 = sha256_file(tuned_checkpoint)

        setting_fields = {
            "setting": setting,
            "seed": int(cfg["seed"]),
            "dropout": float(cfg["dropout"]),
            "lr": float(cfg["lr"]),
            "scheduler": TUNED_SCHEDULER,
            "cosine_t": int(cfg["cosine_t"]),
            "eta_min": TUNED_ETA_MIN,
            "epochs": int(cfg["epochs"]),
            "patience": int(cfg["patience"]),
            "task_batch": int(TUNED_TASK_BATCH[dataset]),
            "selected_universal_variant": selected_variant,
            "selected_universal_checkpoint": str(selected_universal_checkpoint),
            "selected_universal_checkpoint_sha256": selected_universal_checkpoint_sha256,
            "initialization_checkpoint": str(selected_universal_checkpoint),
            "initialization_checkpoint_sha256": selected_universal_checkpoint_sha256,
        }
        out.joinpath("source_provenance.json").write_text(
            json.dumps(
                {
                    "model": "Tuned-UniversalXAS",
                    "dataset": dataset,
                    "setting": setting,
                    "setting_values": {
                        **cfg,
                        "scheduler": TUNED_SCHEDULER,
                        "eta_min": TUNED_ETA_MIN,
                        "batch_size": TUNED_TASK_BATCH[dataset],
                    },
                    "checkpoint": str(tuned_checkpoint),
                    "checkpoint_sha256": tuned_checkpoint_sha256,
                    "monitor": "val_median_mse",
                    "selected_universal_variant": selected_variant,
                    "selected_universal_checkpoint": str(selected_universal_checkpoint),
                    "selected_universal_checkpoint_sha256": selected_universal_checkpoint_sha256,
                    "initialization_checkpoint": str(selected_universal_checkpoint),
                    "initialization_checkpoint_sha256": selected_universal_checkpoint_sha256,
                    "source_encoder_checkpoint": str(SOURCE_ENCODER_CKPT),
                    "source_run": str(SOURCE_RUN),
                    "source_notebook": str(SOURCE_NOTEBOOK),
                    "source_notebook_sha256": sha256_file(SOURCE_NOTEBOOK),
                },
                indent=2,
            ),
            encoding="utf-8",
        )

        tuned_model, _ = load_checkpoint_model(
            tuned_checkpoint,
            hidden_dims=TUNED_HIDDEN_DIMS,
            dropout=float(cfg["dropout"]),
        )
        tuned_val = evaluate_loaded_checkpoint(
            tuned_model,
            tuned_checkpoint,
            "Tuned-UniversalXAS",
            {dataset: split},
            extra_fields=setting_fields,
            split_names=("val",),
        )
        del tuned_model
        tuned_validation_rows.extend(tuned_val.to_dict("records"))
        row = tuned_val.iloc[0]
        print(f"{dataset} ({selected_variant}/{setting}): val_eta={row['val_eta']:.4f}")

if len(tuned_validation_rows) != len(FEFF_DATASETS) * len(TUNED_SWEEP):
    raise RuntimeError(
        f"Expected {len(FEFF_DATASETS) * len(TUNED_SWEEP)} validation candidates, "
        f"got {len(tuned_validation_rows)}"
    )
tuned_validation_candidates = pd.DataFrame(tuned_validation_rows).sort_values(
    ["dataset", "val_eta"], ascending=[True, False]
)
tuned_validation_candidates.to_csv(
    ANALYSIS_DIR / "tuned_validation_candidates.csv", index=False
)

selected_tuned_rows = []
for dataset in FEFF_DATASETS:
    task_candidates = tuned_validation_candidates[
        tuned_validation_candidates["dataset"] == dataset
    ]
    if len(task_candidates) != len(TUNED_SWEEP):
        raise RuntimeError(f"Expected all sweep settings for {dataset}")
    selected_candidate = task_candidates.loc[task_candidates["val_eta"].idxmax()].to_dict()
    selected_checkpoint = Path(selected_candidate["checkpoint"]).resolve()
    selected_model, _ = load_checkpoint_model(
        selected_checkpoint,
        hidden_dims=TUNED_HIDDEN_DIMS,
        dropout=float(selected_candidate["dropout"]),
    )
    selected_test = evaluate_loaded_checkpoint(
        selected_model,
        selected_checkpoint,
        "Tuned-UniversalXAS",
        {dataset: splits[dataset]},
        split_names=("test",),
    )
    del selected_model
    selected_row = dict(selected_candidate)
    selected_row.update(
        {
            key: value
            for key, value in selected_test.iloc[0].to_dict().items()
            if key.startswith("test_")
        }
    )
    selected_tuned_rows.append(selected_row)
    print(
        f"Selected {dataset} ({selected_row['selected_universal_variant']}/{selected_row['setting']}): "
        f"val_eta={selected_row['val_eta']:.4f}, "
        f"test_eta={selected_row['test_eta']:.4f}"
    )

tuned_eval = pd.DataFrame(selected_tuned_rows).sort_values("dataset")
tuned_eval.to_csv(ANALYSIS_DIR / "tuned_eval.csv", index=False)
selected_universal_eval = universal_eval[universal_eval.apply(lambda row: row["universal_variant"] == selected_universal_by_dataset[row["dataset"]]["universal_variant"], axis=1)].copy()
selected_universal_eval["selected_universal_variant"] = selected_universal_eval["universal_variant"]
selected_universal_eval["setting"] = ""
evaluation_summary = pd.concat([selected_universal_eval, tuned_eval], ignore_index=True, sort=False)
evaluation_summary.to_csv(ANALYSIS_DIR / "evaluation_summary.csv", index=False)


Seed set to 145
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fullTrainingCopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff/tunedUniversalXAS/Ti_FEFF/feff_plateau_lr5e-4_do010_seed45/cosT500_lr3e-4_do0p1_p60 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type     | Params | Mode  | FLOPs
---------------------------------------------------
0 | loss  | MSELoss  | 0      | train | 0    
1 | model | XASBlock | 639 K  | train | 0    
---------------------------------------------------
639 K  

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

## 9. Completion and provenance

`analysis/universal_eval.csv` contains both Universal candidates across all eight FEFF datasets. `analysis/universal_selected_by_dataset.csv` records the source selected by validation eta and reports its test eta without using test for selection.

`analysis/tuned_validation_candidates.csv` contains all 56 validation-only candidates with their selected Universal source. `analysis/tuned_eval.csv` contains the eight selected tuned checkpoints. `analysis/evaluation_summary.csv` contains the eight selected Universal rows and eight selected tuned rows.

In [ ]:
universal_checkpoint_rows = [{"universal_variant":c["name"],"universal_seed":int(c["seed"]),"checkpoint":str(universal_checkpoints[c["name"]]),"checkpoint_sha256":sha256_file(universal_checkpoints[c["name"]])} for c in UNIVERSAL_CANDIDATES]
source_manifest.update({"universal_candidates":universal_manifest_candidates,"universal_checkpoints":universal_checkpoint_rows,"universal_selection_policy":universal_selection_policy,"universal_selected_by_dataset":selected_universal_rows,"tuned_training_monitor":"val_median_mse","tuned_selection_metric":"val_eta","tuned_selection_policy":"Select the highest validation eta per dataset. Evaluate test once for each selected checkpoint.","tuned_validation_only_candidates":True,"tuned_sweep":TUNED_SWEEP,"tuned_scheduler":TUNED_SCHEDULER,"tuned_eta_min":TUNED_ETA_MIN,"evaluation_files":analysis_files})
manifest_path.write_text(json.dumps(source_manifest,indent=2),encoding="utf-8")
print(f"Saved manifest: {manifest_path}")
print(f"Saved Universal evaluation: {ANALYSIS_DIR / 'universal_eval.csv'}")
print(f"Saved Universal selection: {ANALYSIS_DIR / 'universal_selected_by_dataset.csv'}")
print(f"Saved validation candidates: {ANALYSIS_DIR / 'tuned_validation_candidates.csv'}")
print(f"Saved evaluation summary: {ANALYSIS_DIR / 'evaluation_summary.csv'}")
expected_candidate_count = len(FEFF_DATASETS) * len(TUNED_SWEEP)
if len(tuned_validation_rows) != expected_candidate_count: raise RuntimeError(f"Expected {expected_candidate_count} tuned validation candidates: {len(tuned_validation_rows)}")
if len(tuned_eval) != len(FEFF_DATASETS) or set(tuned_eval["dataset"]) != set(FEFF_DATASETS): raise RuntimeError("Expected one selected tuned run for each element")
if len(set(tuned_eval["checkpoint"])) != len(FEFF_DATASETS): raise RuntimeError("Selected tuned checkpoints must be unique per dataset")
if len(universal_eval) != len(UNIVERSAL_CANDIDATES) * len(FEFF_DATASETS): raise RuntimeError("Universal evaluation must contain every candidate for every dataset")
if len(universal_selected_by_dataset) != len(FEFF_DATASETS): raise RuntimeError("Universal selection must contain one source per dataset")
run_complete = {"status":"complete","completion_label":"balanced_new_scheduler_universal_tuned_feff_complete","sampler_info":source_manifest["sampler_info"],"universal_candidates":universal_manifest_candidates,"universal_checkpoints":universal_checkpoint_rows,"universal_selection_policy":universal_selection_policy,"universal_selected_by_dataset":selected_universal_rows,"tuned_sweep":TUNED_SWEEP,"tuned_selection_policy":"Highest validation eta per dataset. Test runs once for each selected checkpoint.","tuned_checkpoints":[{"dataset":r["dataset"],"setting":r["setting"],"selected_universal_variant":r["selected_universal_variant"],"selected_universal_checkpoint":r["selected_universal_checkpoint"],"selected_universal_checkpoint_sha256":r["selected_universal_checkpoint_sha256"],"checkpoint":r["checkpoint"],"checkpoint_sha256":r["checkpoint_sha256"]} for r in selected_tuned_rows],"tuned_run_count":len(selected_tuned_rows),"tuned_candidate_count":len(tuned_validation_rows),"evaluation_files":analysis_files}
(RUN_DIR / "RUN_COMPLETE.json").write_text(json.dumps(run_complete,indent=2),encoding="utf-8")
print(f"Saved completion marker: {RUN_DIR / 'RUN_COMPLETE.json'}")
final_columns = ["model","dataset","universal_variant","selected_universal_variant","setting","val_eta","test_eta","val_median_mse","test_median_mse"]
display(evaluation_summary[final_columns].sort_values(["dataset","model"]).style.format({"val_eta":"{:.3f}","test_eta":"{:.3f}","val_median_mse":"{:.5f}","test_median_mse":"{:.5f}"}))